# Câu 6: Tiền xử lý các thuộc tính thiếu dữ liệu và xóa bỏ các thuộc tính không cần thiết

Phần 1 – Bài 2 (trang 15–17): Yêu cầu lập trình trên dữ liệu Titanic

In [1]:
SV = "Họ tên: Mai Quốc Đại - MSSV: 6451071013 - Lớp: CQ.64.CNTT"
DATA = "../../data/"  # thư mục chứa dữ liệu
print(SV)

Họ tên: Mai Quốc Đại - MSSV: 6451071013 - Lớp: CQ.64.CNTT


In [8]:
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

### Chuẩn bị: thực hiện lại các bước của các câu trước

In [ ]:
df = pd.read_csv(DATA + 'titanic.csv', dtype={'age': np.float64, 'sibsp': np.int64, 'parch': np.int64})

df['relatives'] = df['sibsp'] + df['parch']
# not_alone = 1 nếu có người thân đi cùng, 0 nếu đi một mình
df['not_alone'] = (df['relatives'] > 0).astype(int)

---
## Thực hiện yêu cầu

- `body`, `cabin`, `boat`, `home.dest` thiếu rất nhiều và không ảnh hưởng nhiều tới khả năng sống sót → xóa. Riêng `cabin`, ký tự đầu là boong (deck) nên tách ra giữ lại trước khi xóa.
- `ticket` có quá nhiều giá trị khác nhau → xóa.
- `age`: điền ngẫu nhiên trong khoảng [mean − std, mean + std]; `embarked`: điền bằng mode; `fare`: điền bằng trung vị.

In [10]:
print(SV)
df['ticket'].describe()

Họ tên: Mai Quốc Đại - MSSV: 6451071013 - Lớp: CQ.64.CNTT


count         1309
unique         929
top       CA. 2343
freq            11
Name: ticket, dtype: object

In [11]:
# Tách boong (deck) từ cabin trước khi xóa cabin
deck = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7, "U": 8}
df['cabin'] = df['cabin'].fillna('U0')
df['deck'] = df['cabin'].map(lambda x: re.compile('([a-zA-Z]+)').search(x).group())
df['deck'] = df['deck'].map(deck).fillna(0).astype(int)   # 'T' (1 vé lạ) -> 0

# Xóa các thuộc tính không cần thiết
df = df.drop(['cabin', 'ticket', 'boat', 'body', 'home.dest'], axis=1)

# Điền tuổi bằng số ngẫu nhiên trong khoảng [mean - std, mean + std]
np.random.seed(42)                       # cố định để kết quả lặp lại được
mean, std = df['age'].mean(), df['age'].std()
is_null = df['age'].isnull().sum()
rand_age = np.random.randint(int(mean - std), int(mean + std), size=is_null)
age_slice = df['age'].copy()
age_slice[np.isnan(age_slice)] = rand_age
df['age'] = age_slice.astype(int)

# Cổng lên tàu: điền bằng giá trị phổ biến nhất (mode)
common_value = df['embarked'].mode()[0]
df['embarked'] = df['embarked'].fillna(common_value)

# Giá vé: chỉ thiếu 1 giá trị -> điền bằng trung vị giá vé
df['fare'] = df['fare'].fillna(df['fare'].median())

In [12]:
df.isnull().sum()

pclass       0
survived     0
name         0
sex          0
age          0
sibsp        0
parch        0
fare         0
embarked     0
relatives    0
not_alone    0
deck         0
dtype: int64

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   pclass     1309 non-null   int64  
 1   survived   1309 non-null   int64  
 2   name       1309 non-null   str    
 3   sex        1309 non-null   str    
 4   age        1309 non-null   int64  
 5   sibsp      1309 non-null   int64  
 6   parch      1309 non-null   int64  
 7   fare       1309 non-null   float64
 8   embarked   1309 non-null   str    
 9   relatives  1309 non-null   int64  
 10  not_alone  1309 non-null   int64  
 11  deck       1309 non-null   int64  
dtypes: float64(1), int64(8), str(3)
memory usage: 122.8 KB


**Nhận xét:**

- Đã tách `deck` từ `cabin` (U = không rõ boong) rồi xóa `cabin`, `ticket`, `boat`, `body`, `home.dest`.
- `age` được điền bằng số ngẫu nhiên trong khoảng [mean − std, mean + std]; cố định `np.random.seed(42)` để chạy lại cho cùng kết quả.
- `embarked` thiếu 2 giá trị, điền bằng `S` (mode); `fare` thiếu 1 giá trị, điền bằng trung vị.
- Sau tiền xử lý không còn giá trị rỗng; dữ liệu còn 12 thuộc tính.